# Aprendizado de Máquina — Lista prática E2

## Redução de Dimensionalidade (PCA e t-SNE)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Duas perguntas nesta lista, e nenhuma delas é "qual método é melhor".

A primeira é sobre **o que cada método preserva** — PCA e t-SNE otimizam coisas
diferentes, e medir as duas coisas mostra que cada um ganha exatamente onde o
outro perde. A segunda revisita a Aula 06:

> **ajustar o PCA no conjunto todo, antes de separar treino e validação, usa
> informação do teste. Isso infla o desempenho medido? Meça antes de responder.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — PCA pela SVD, à mão

A Lista Teórica E2 mostrou que, se $X = UDV^\top$ é a decomposição em valores
singulares dos dados **centrados**, então as colunas de $V$ são as componentes
principais, os autovalores são $d_j^2/(n-1)$, e as coordenadas são as colunas de
$UD$.

Faça as três coisas e confira contra o `scikit-learn`. O banco é o `digits`:
1 797 imagens $8\times 8$ de dígitos manuscritos, achatadas em 64 colunas.

In [ ]:
dados = load_digits()
X, y = dados.data, dados.target
print(f"digits: {X.shape}, {len(set(y))} classes")

X_c = X - X.mean(axis=0)                                         # (a) centrar e obrigatorio

U, D, Vt = np.linalg.svd(X_c, full_matrices=False)
autovalores = D ** 2 / (len(X) - 1)                              # (b) divisor n-1, como o sklearn

pca = PCA().fit(X_c)

print(f"5 primeiros autovalores (SVD)    : {autovalores[:5].round(4)}")
print(f"5 primeiros (sklearn)            : {pca.explained_variance_[:5].round(4)}")
print(f"diferenca maxima nos 60 primeiros: "
      f"{np.abs(autovalores[:60] - pca.explained_variance_[:60]).max():.2e}")

In [ ]:
Z_svd = (U * D)[:, :2]                                           # (a) as coordenadas sao as colunas de UD
Z_skl = pca.transform(X_c)[:, :2]

# os autovetores sao definidos a menos de sinal, entao comparamos os modulos
print(f"diferenca maxima nas coordenadas: "
      f"{np.abs(np.abs(Z_svd) - np.abs(Z_skl)).max():.2e}")

Deve imprimir os autovalores `[179.0069 163.7177 141.7884 101.1004 69.5132]` nas
duas linhas, com diferença máxima da ordem de $10^{-13}$, e o mesmo para as
coordenadas.

Duas armadilhas que a lacuna (a) e o comentário cobrem:

**Centrar é obrigatório.** Sem subtrair a média, a "primeira componente" acaba
apontando para o centro de massa da nuvem, e não para a direção de maior
dispersão. O `PCA` do `scikit-learn` centra sozinho — é por isso que ele funciona
mesmo se você lhe der `X` em vez de `X_c` — mas a SVD não.

**O divisor é $n-1$**, e não $n$ como na Lista Teórica E2. É convenção: o
`scikit-learn` usa a variância amostral não-viesada. A escolha muda os
autovalores por um fator $n/(n-1)$ e **não muda** nem os autovetores nem as
proporções de variância explicada.

---
## Exercício 2 — quantas componentes bastam

Cada imagem tem 64 pixels, mas os pixels de uma imagem de dígito são fortemente
correlacionados: as bordas são quase sempre pretas, e traços vizinhos aparecem
juntos. Quanta dimensão de verdade existe ali?

In [ ]:
acumulada = np.cumsum(pca.explained_variance_ratio_)             # (a)

for alvo in (0.80, 0.90, 0.95, 0.99):
    n_comp = int(np.argmax(acumulada >= alvo)) + 1               # (b) o primeiro que atinge o alvo
    print(f"{int(alvo * 100)}% da variancia: {n_comp:2d} componentes de {X.shape[1]}")

print(f"\nPCA(n_components=0.90) devolve: {PCA(n_components=0.90).fit(X_c).n_components_}")   # (c)

Deve imprimir:

```
80% da variancia: 13 componentes de 64
90% da variancia: 21 componentes de 64
95% da variancia: 29 componentes de 64
99% da variancia: 41 componentes de 64

PCA(n_components=0.90) devolve: 21
```

**Vinte e uma das 64 colunas carregam 90% da variância** — as 43 restantes,
somadas, valem 10%. E note o rendimento decrescente, que é o padrão em dados
reais: as primeiras 13 compram 80%, as 8 seguintes compram mais 10%, e as 20
seguintes compram só 9%.

A última linha mostra a conveniência do `float`: pedir `n_components=0.90` faz o
`scikit-learn` chegar sozinho ao 21. Num `Pipeline` com validação cruzada, esse
número pode até variar entre dobras — e é isso que se quer, porque o critério
("reter 90%") é que deve ser constante, não a contagem.

---
## Exercício 3 — o que cada método preserva

Agora compare PCA e t-SNE por **duas** medidas diferentes:

- **estrutura global**: a correlação de Spearman entre as distâncias no espaço
  original e as distâncias no mapa, sobre todos os pares;
- **estrutura local**: a fração dos 10 vizinhos mais próximos de cada ponto no
  espaço original que continuam entre os 10 mais próximos no mapa.

Use uma subamostra de 800 pontos — o t-SNE tem custo quadrático.

In [ ]:
sub = np.random.default_rng(2026).choice(len(X), 800, replace=False)
X_s, y_s = X[sub], y[sub]

Z_pca = PCA(n_components=2).fit_transform(X_s)
Z_tsne = TSNE(n_components=2, perplexity=30, random_state=2026,
              init="pca").fit_transform(X_s)                       # (a)

dist_alto = pdist(X_s)


def preservacao_vizinhanca(X_alto, Z_baixo, k=10):
    viz_alto = NearestNeighbors(n_neighbors=k + 1).fit(X_alto) \
        .kneighbors(X_alto, return_distance=False)[:, 1:]          # [:, 1:] descarta o proprio ponto
    viz_baixo = NearestNeighbors(n_neighbors=k + 1).fit(Z_baixo) \
        .kneighbors(Z_baixo, return_distance=False)[:, 1:]
    return np.mean([len(set(a) & set(b)) / k                     # (a) vizinhos em comum
                    for a, b in zip(viz_alto, viz_baixo)])


print("          correlacao global    preservacao local")
for nome, Z in (("PCA", Z_pca), ("t-SNE", Z_tsne)):
    global_ = spearmanr(dist_alto, pdist(Z)).correlation         # (b)
    local = preservacao_vizinhanca(X_s, Z)
    print(f"  {nome:6s}      {global_:.4f}              {local:.4f}")

Deve imprimir:

```
          correlacao global    preservacao local
  PCA         0.5838              0.1954
  t-SNE       0.4634              0.6466
```

**Cada um ganha exatamente onde o outro perde**, e a diferença de magnitude é
reveladora.

Na **estrutura global** o PCA leva por pouco (0,58 contra 0,46). Faz sentido: o
PCA é uma projeção linear, então distâncias grandes no espaço original tendem a
continuar grandes no mapa — ele nunca aproxima artificialmente dois pontos
distantes.

Na **estrutura local** o t-SNE leva de lavada: **0,65 contra 0,20**, mais de três
vezes. Dos 10 vizinhos verdadeiros de cada ponto, o t-SNE preserva 6 ou 7 e o PCA
preserva 2. E isso é exatamente o que a função-objetivo dele otimiza — o
Exercício 3(a) da Lista Teórica E2 mostrou que a divergência de KL na direção
usada pune severamente separar vizinhos verdadeiros e quase não pune aproximar
pontos distantes.

**A leitura prática.** Se você vai *olhar* o diagrama para descobrir se há grupos,
o t-SNE mostra muito mais. Se você vai *medir* alguma coisa a partir das
coordenadas — distância entre grupos, tamanho de agrupamento, qualquer conta —
nenhum dos dois é confiável, e o t-SNE menos ainda: aquele $0{,}46$ de correlação
global significa que quase metade da informação sobre distâncias se perdeu.

> **Sua vez.** Desenhe os dois mapas lado a lado, coloridos pelo dígito
> verdadeiro (`c=y_s`). Em qual dos dois os dez algarismos aparecem separados?

---
## Exercício 4 — o PCA fora da dobra é vazamento grave?

Ajustar o PCA no conjunto completo, antes de separar treino e validação, **usa
informação do teste**: as direções principais dependem de todos os pontos. Pela
intuição, isso parece grave — o PCA aprende de todas as colunas ao mesmo tempo.

Mas o critério da Aula 06 não é "quanto a etapa aprende", é **"a etapa olha o
$Y$?"**. O PCA não olha. Vamos medir o efeito, no cenário mais cruel possível:
$y$ de ruído puro, em que o $R^2$ verdadeiro é zero e qualquer inflação fica
visível.

In [ ]:
cv = skm.KFold(5, shuffle=True, random_state=0)

print("       cenario           PCA fora    PCA dentro   diferenca")
for n, d, k in [(60, 200, 10), (60, 1000, 20), (100, 500, 15), (200, 2000, 30)]:
    rng = np.random.default_rng(2026)
    fora, dentro = [], []

    for _ in range(20):
        X_r = rng.normal(size=(n, d))
        y_r = rng.normal(size=n)                                 # (a) NENHUMA relacao com X

        # ERRADO: o PCA ve o conjunto todo, depois valida
        # svd_solver="full" torna o resultado deterministico: com d grande, o
        # padrao "auto" escolhe o solucionador randomizado, que varia entre execucoes
        X_p = PCA(n_components=k, svd_solver="full").fit_transform(X_r)   # (b)
        fora.append(skm.cross_val_score(skl.LinearRegression(), X_p, y_r,
                                        cv=cv, scoring="r2").mean())

        # CERTO: o PCA e um passo do pipeline, refeito em cada dobra
        tubo = Pipeline([("pca", PCA(n_components=k, svd_solver="full")),   # (c)
                         ("mqo", skl.LinearRegression())])
        dentro.append(skm.cross_val_score(tubo, X_r, y_r, cv=cv, scoring="r2").mean())

    print(f"  n={n:3d} d={d:4d} k={k:2d}    {np.mean(fora):+.4f}     "
          f"{np.mean(dentro):+.4f}     {np.mean(fora) - np.mean(dentro):+.4f}")   # (d)

Deve imprimir:

```
       cenario           PCA fora    PCA dentro   diferenca
  n= 60 d= 200 k=10    -0.5934     -0.2167     -0.3767
  n= 60 d=1000 k=20    -1.1846     -0.1614     -1.0232
  n=100 d= 500 k=15    -0.4386     -0.1109     -0.3277
  n=200 d=2000 k=30    -0.3306     -0.0517     -0.2789
```

**O PCA fora da dobra não infla nada. Ele piora.** Nas quatro configurações a
diferença é negativa, de $-0{,}28$ a $-1{,}02$.

Compare com o Exercício 1 da Lista prática 06, o mesmo experimento com **seleção
de variáveis** no lugar do PCA: lá o $R^2$ ia de $-0{,}70$ (certo) para
$+0{,}41$ (errado), uma inflação de mais de uma unidade. Aqui o erro anda na
direção oposta.

A explicação está no critério: **o PCA nunca vê o $y$.** As direções principais
são calculadas só a partir de $X$, então não existe canal pelo qual o ruído da
dobra de validação possa entrar na construção das covariáveis. Sem esse canal,
não há como fabricar acerto.

E por que ele *piora*? Porque componentes calculados sobre o conjunto todo não
são os componentes ótimos de nenhuma dobra de treino em particular. O modelo
ajustado numa dobra recebe uma base ligeiramente desalinhada com os dados que ele
viu — um pequeno prejuízo, que aparece amplificado justamente porque não há sinal
nenhum a recuperar.

**Consequência para a hierarquia da Aula 06.** O PCA pertence à categoria
**leve**, junto com a padronização e a imputação pela média — não à categoria
grave, junto com a seleção de variáveis e o vazamento por grupo. A conduta
prática não muda (ele vai para o `Pipeline`, porque custa uma linha), mas o
lugar dele na lista de preocupações, sim.

> **Sua vez.** Repita o primeiro cenário trocando `y_r` por um $y$ que **dependa**
> de `X_r` — por exemplo, `X_r @ beta + ruido` com `beta` esparso. A diferença
> entre fazer o PCA fora e dentro continua negativa?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | a PCA pela SVD à mão bate o `scikit-learn` até $10^{-13}$, em autovalores e coordenadas |
| 2 | 21 das 64 colunas do `digits` carregam 90% da variância |
| 3 | o PCA preserva melhor a estrutura **global** (0,5838 contra 0,4634) |
| 3 | o t-SNE preserva **três vezes mais** vizinhança local (0,6466 contra 0,1954) |
| 4 | o PCA fora da dobra **piora** o $R^2$ em 0,28 a 1,02 — nunca infla |

**A seguir.** A Aula E3 fecha o curso aplicando quase tudo a um problema real de
texto: representar mensagens como vetores, classificar, e escolher a métrica
certa para um problema desbalanceado.